import sparkocr
import pyspark
import sys
import sparknlp_jsl
from pyspark.sql import SparkSession
from sparkocr import start
import os
import base64
from sparkocr.transformers import *
from pyspark.ml import PipelineModel
from pyspark.sql import functions as F
from sparkocr.enums import *
from sparkocr.utils import display_images

from sparknlp.annotator import *
from sparknlp_jsl.annotator import *
from sparknlp.base import *

In [2]:
spark = sparkocr.start(
    secret=os.environ.get('SPARK_OCR_SECRET', ''),
    nlp_version=os.environ.get('PUBLIC_VERSION', ''),
    nlp_secret=os.environ.get('SECRET', ''),
    nlp_internal=os.environ.get('JSL_VERSION', '')
)
spark

Spark version: 3.1.1
Spark NLP version: 5.5.3
Spark NLP for Healthcare version: 6.0.2
Spark OCR version: 6.0.0

:: loading settings :: url = jar:file:/usr/local/lib/python3.8/dist-packages/pyspark/jars/ivy-2.4.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-35891c52-6419-4276-905e-cb07a41548ea;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp_2.12;6.0.2 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in central
	found com.amazonaws#jmespath-java;1.12.500 in central
	found com.g

25/06/24 20:00:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/24 20:00:45 ERROR Inbox: Ignoring error
java.lang.NullPointerException
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$register(BlockManagerMasterEndpoint.scala:524)
	at org.apache.spark.storage.BlockManagerMasterEndpoint$$anonfun$receiveAndReply$1.applyOrElse(BlockManagerMasterEndpoint.scala:116)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:103)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:213)
	at org.apache.spark.rpc.netty.Inbox.process(Inbox.scala:100)
	at org.apache.spark.rpc.netty.MessageLoop.org$apache$spark$rpc$netty$MessageLoop

In [3]:
def deidentification_nlp_pipeline(input_column, prefix = ""):
    document_assembler = DocumentAssembler() \
        .setInputCol(input_column) \
        .setOutputCol(prefix + "document")

    # Sentence Detector annotator, processes various sentences per line
    sentence_detector = SentenceDetector() \
        .setInputCols([prefix + "document"]) \
        .setOutputCol(prefix + "sentence")

    tokenizer = Tokenizer() \
        .setInputCols([prefix + "sentence"]) \
        .setOutputCol(prefix + "token")

    # Clinical word embeddings
    word_embeddings = WordEmbeddingsModel.pretrained("embeddings_clinical", "en", "clinical/models") \
        .setInputCols([prefix + "sentence", prefix + "token"]) \
        .setOutputCol(prefix + "embeddings")
    # NER model trained on i2b2 (sampled from MIMIC) dataset
    clinical_ner = MedicalNerModel.pretrained("ner_deid_large", "en", "clinical/models") \
        .setInputCols([prefix + "sentence", prefix + "token", prefix + "embeddings"]) \
        .setOutputCol(prefix + "ner")

    custom_ner_converter = NerConverterInternal() \
        .setInputCols([prefix + "sentence", prefix + "token", prefix + "ner"]) \
        .setOutputCol(prefix + "ner_chunk") \
        .setWhiteList(['NAME', 'AGE', 'CONTACT', 'LOCATION', 'PROFESSION', 'PERSON', 'DATE'])

    nlp_pipeline = Pipeline(stages=[
            document_assembler,
            sentence_detector,
            tokenizer,
            word_embeddings,
            clinical_ner,
            custom_ner_converter
        ])
    empty_data = spark.createDataFrame([[""]]).toDF(input_column)
    nlp_model = nlp_pipeline.fit(empty_data)
    return nlp_model

In [4]:
# Read dicom as image
dicom_to_image = DicomToImage() \
    .setInputCol("content") \
    .setOutputCol("image_raw") \
    .setMetadataCol("metadata") \
    .setDeIdentifyMetadata(True)

adaptive_thresholding = ImageAdaptiveThresholding() \
    .setInputCol("image_raw") \
    .setOutputCol("corrected_image") \
    .setBlockSize(47) \
    .setOffset(4) \
    .setKeepInput(True)

# Extract text from image
ocr = ImageToText() \
    .setInputCol("corrected_image") \
    .setOutputCol("text")

# Found coordinates of sensitive data
position_finder = PositionFinder() \
    .setInputCols("ner_chunk") \
    .setOutputCol("coordinates") \
    .setPageMatrixCol("positions")

# Found sensitive data using DeIdentificationModel
deidentification_rules = DeIdentificationModel.pretrained("deidentify_rb_no_regex", "en", "clinical/models") \
    .setInputCols(["metadata_sentence", "metadata_token", "metadata_ner_chunk"]) \
    .setOutputCol("deidentified_metadata_raw")

finisher = Finisher() \
    .setInputCols(["deidentified_metadata_raw"]) \
    .setOutputCols("deidentified_metadata") \
    .setOutputAsArray(False) \
    .setValueSplitSymbol("") \
    .setAnnotationSplitSymbol("")

# Draw filled rectangle for hide sensitive data
drawRegions = ImageDrawRegions()  \
    .setInputCol("image_raw")  \
    .setInputRegionsCol("coordinates")  \
    .setOutputCol("image_with_regions")  \
    .setFilledRect(True) \
    .setRectColor(Color.black)

# Store image back to Dicom document
imageToDicom = ImageToDicom() \
    .setInputCol("image_with_regions") \
    .setOutputCol("dicom")

# OCR pipeline
deid_pipeline = PipelineModel(stages=[
    dicom_to_image,
    adaptive_thresholding,
    ocr,
    deidentification_nlp_pipeline(input_column="text"),
    position_finder,
    drawRegions,
    #imageToDicom  # Commented for able to demonstrate intermidiate results before aggregation
])

deidentify_rb_no_regex download started this may take some time.
[ | ]

25/06/24 20:00:56 WARN ApacheUtils: NoSuchMethodException was thrown when disabling normalizeUri. This indicates you are using an old version (< 4.5.8) of Apache http client. It is recommended to use http client version >= 4.5.9 to avoid the breaking change introduced in apache client 4.5.7 and the latency in exception handling. See https://github.com/aws/aws-sdk-java/issues/1919 for more information


[ / ]deidentify_rb_no_regex download started this may take some time.
Approximate size to download 8.9 KB
Download done! Loading the resource.
[OK!]
embeddings_clinical download started this may take some time.
Approximate size to download 1.6 GB
[ | ]

25/06/24 20:01:00 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
25/06/24 20:01:00 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
25/06/24 20:01:00 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


embeddings_clinical download started this may take some time.
Approximate size to download 1.6 GB
Download done! Loading the resource.
[OK!]
ner_deid_large download started this may take some time.
[ | ]

25/06/24 20:01:05 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
25/06/24 20:01:05 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


ner_deid_large download started this may take some time.
Approximate size to download 14.1 MB
Download done! Loading the resource.
[ / ]

2025-06-24 20:01:08.677336: I external/org_tensorflow/tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-24 20:01:08.786477: W external/org_tensorflow/tensorflow/core/common_runtime/colocation_graph.cc:1218] Failed to place the graph without changing the devices of some resources. Some of the operations (that had to be colocated with resource generating operations) are not supported on the resources' devices. Current candidate devices are [
  /job:localhost/replica:0/task:0/device:CPU:0].
See below for details of this colocation group:
Colocation Debug Info:
Colocation group had the following types and supported devices: 
Root Member(assigned_device_name_index_=-1 requested_device_name_='/device:GPU:0' assigned_device_nam

[OK!]


In [5]:
from pathlib import Path

from pyspark.ml import Pipeline
from pyspark.sql import SparkSession
from pyspark.sql.functions import input_file_name

from sparkocr.transformers import *
from sparkocr.enums import *
from sparkocr.utils import display_image, display_images
from sparkocr.metrics import score
from sparknlp.annotator import *
from sparknlp_jsl.annotator import *
from sparknlp.base import *
import sparknlp_jsl

from sparkocr.transformers import *
from sparkocr.utils import display_image, to_pil_image

In [6]:
# Read Pdf as image
pdf_to_image = PdfToImage()\
    .setInputCol("content")\
    .setOutputCol("image_raw")\
    .setResolution(400)

image_to_pdf = ImageToPdf() \
    .setInputCol("image_with_regions") \
    .setOutputCol("pdf_bytes")

# Extract text from image
ocr = ImageToText() \
    .setInputCol("image_raw") \
    .setOutputCol("text") \
    .setIgnoreResolution(False) \
    .setPageIteratorLevel(PageIteratorLevel.SYMBOL) \
    .setPageSegMode(PageSegmentationMode.SPARSE_TEXT) \
    .setWithSpaces(True) \
    .setConfidenceThreshold(70)

hocr = ImageToHocr() \
    .setInputCol("image_raw") \
    .setOutputCol("hocr") \
    .setIgnoreResolution(False) \
    .setOcrParams(["preserve_interword_spaces=0"])\
    .setPageIteratorLevel(PageIteratorLevel.SYMBOL)\
    .setPageSegMode(PageSegmentationMode.SPARSE_TEXT) \

# Found coordinates of sensitive data
position_finder = PositionFinder() \
    .setInputCols("ner_chunk") \
    .setOutputCol("coordinates") \
    .setPageMatrixCol("positions")

# Draw filled rectangle for hide sensitive data
drawRegions = ImageDrawRegions()  \
    .setInputCol("image_raw")  \
    .setInputRegionsCol("coordinates")  \
    .setOutputCol("image_with_regions")  \
    .setFilledRect(True) \
    .setRectColor(Color.black)

# OCR pipeline
deid_pipeline = PipelineModel(stages=[
    pdf_to_image,
    ocr,
    hocr,
    deidentification_nlp_pipeline(input_column="text"),
    position_finder,
    drawRegions
])

embeddings_clinical download started this may take some time.
Approximate size to download 1.6 GB
[ | ]

25/06/24 20:01:12 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


[OK!]
ner_deid_large download started this may take some time.
[OK!]


In [7]:
input_folder  = "/data/input"
output_folder = "/data/output"
os.makedirs(output_folder, exist_ok=True)

pdfs = (
    spark.read
         .format("binaryFile")
         .load(f"{input_folder}/*.pdf")
         .withColumn("orig_path", input_file_name())
)
pdfs = pdfs.coalesce(4) # only 4 partitions at a time

processed = deid_pipeline.transform(pdfs)
result = (
    image_to_pdf
      .transform(processed)
      .withColumnRenamed("path", "orig_path")
      .select("orig_path", "pdf_bytes")
)

for row in result.collect():
    src = Path(row.orig_path)
    out = Path(output_folder) / src.name
    with open(out, "wb") as f:
        f.write(row.pdf_bytes)

print(f"Wrote de-ID’d PDFs to {output_folder}")

25/06/24 20:01:19 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
Detected 2123 diacritics
Detected 1490 diacritics
Detected 906 diacritics
Detected 1519 diacritics
Detected 312 diacritics
Detected 1502 diacritics
Detected 2279 diacritics
Detected 1253 diacritics
Detected 1741 diacritics
Detected 1758 diacritics
Detected 277 diacritics
Detected 2109 diacritics                                            (0 + 1) / 1]
Detected 913 diacritics
Detected 206 diacritics
Detected 293 diacritics
Detected 1249 diacritics
Detected 242 diacritics
Detected 2289 diacritics
Detected 636 diacritics
Detected 1315 diacritics
Detected 1057 diacritics
Detected 1026 diacritics
Detected 2279 diacritics                                          (0 + 12) / 12]
Detected 1315 diacritics
Detected 2289 diacritics
Detected 206 diacritics
Detected 1253 diacritics
Detected 636 diacritics
Detected 1519 diac

25/06/24 20:03:40 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 74, end: 101, result: Officia
az
STANFORD HOSPITAL), index: 2
25/06/24 20:03:40 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 439, end: 447, result: Je
CENTER), index: 8
25/06/24 20:03:40 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 850, end: 866, result: STANFORD
Facility), index: 19
25/06/24 20:03:40 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 1021, end: 1036, result: Ste 205
Monterey), index: 22
25/06/24 20:03:40 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 1053, end: 1081, result: (834) 373-5465
Comidentiality), index: 24
Detected 1758 diacritics                                          (2 + 10) / 12]
Detected 2109 diacritics
Detected 242 diacritics
Detected 312 diacritics
25/06/24 20:03:43 ERROR PositionFinder: PositionFinder unmatched:::Annotatio

25/06/24 20:03:49 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 103, end: 127, result: Pimentel, Teri M
Stanford), index: 3
25/06/24 20:03:49 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 103, end: 127, result: Pimentel, Teri M
Stanford), index: 3
25/06/24 20:03:49 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 103, end: 127, result: Pimentel, Teri M
Stanford), index: 3
25/06/24 20:03:49 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 82, end: 115, result: Rees
az
Stanford
STANFORD HOSPITAL), index: 2
25/06/24 20:03:50 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 122, end: 150, result: Pimentel, Teri M
MHEALTN CARE), index: 3
25/06/24 20:03:50 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 82, end: 115, result: Rees
az
Stanford
STANFORD HOSPITAL), index: 2
25/06/24 20:03:50 ERROR Position

25/06/24 20:03:54 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 103, end: 120, result: Pimentel, Teri M
o), index: 3
25/06/24 20:03:54 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 85, end: 122, result: Stan ford
CANCER CENTER SOUTH Pimentel), index: 2
25/06/24 20:03:54 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 2038, end: 2072, result: 1039
Kong,
HOSPITAL
Christina
Drive), index: 12
25/06/24 20:03:54 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 2127, end: 2148, result: Hillview
HILLVIEW
Kong), index: 13
25/06/24 20:03:54 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 2190, end: 2202, result: Christina
Ave), index: 17
25/06/24 20:03:54 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 85, end: 122, result: Stan ford
CANCER CENTER SOUTH Pimentel), index: 2
25/06/24 20:03:54 ERROR Positio

Detected 1741 diacritics================================>         (10 + 2) / 12]
25/06/24 20:03:57 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 80, end: 105, result: Stanford
STANFORD HOSPITAL), index: 3
25/06/24 20:03:57 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 107, end: 125, result: Pimentel, Teri M
SS), index: 4
25/06/24 20:03:57 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 80, end: 105, result: Stanford
STANFORD HOSPITAL), index: 3
25/06/24 20:03:57 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 107, end: 125, result: Pimentel, Teri M
SS), index: 4
25/06/24 20:03:57 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 80, end: 105, result: Stanford
STANFORD HOSPITAL), index: 3
25/06/24 20:03:57 ERROR PositionFinder: PositionFinder unmatched:::Annotation(type: chunk, begin: 107, end: 125, result: Pimentel, Teri M
SS)

Wrote de-ID’d PDFs to /data/output
